<a href="https://colab.research.google.com/github/darkreztadonduruman-svg/github-demo/blob/main/Total_completion_purwadhika_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Force upgrade core dependencies together to eliminate library version mismatch conflicts
!pip install --upgrade langchain-core langchain-google-genai langchain langgraph streamlit langfuse pyngrok qdrant-client -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [5]:
import os
import time
import urllib.request
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types  # Import types for the dimensionality configuration
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# 1. Extract your keys safely out of secret memory
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

def run_phase1_unified_pipeline(file_path, collection_name="imdb_movie_dataset", force_reset=True):
    print(f"📖 Step A: Verifying data file presence...")

    if not os.path.exists(file_path):
        print(f"📥 File '{file_path}' was missing from Colab runtime memory. Downloading standard copy now...")
        url = "https://githubusercontent.com"
        try:
            urllib.request.urlretrieve(url, file_path)
            print("✅ Download finished successfully!")
        except Exception as e:
            raise ValueError(f"Failed to automatically download dataset. Error: {e}")

    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported format! Please use a standard .csv or .xlsx file.")

    total_records = len(df)
    print(f"🎬 Total rows detected in dataset: {total_records} movies.")

    print("🤖 Step B: Translating spreadsheet rows into descriptive text sentences...")
    text_list = []
    for index, row in df.iterrows():
        row_text = ", ".join([f"{column}: {str(value)}" for column, value in row.items()])
        text_list.append((index, row_text))

    print("🌐 Step C: Establishing connection to your Qdrant Cloud Cluster...")
    qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    ai_client = genai.Client(api_key=GOOGLE_API_KEY, http_options={"api_version": "v1"})

    collection_exists = qdrant_client.collection_exists(collection_name)

    if collection_exists and force_reset:
        print(f"⚠️ force_reset=True: Wiping out old mismatched '{collection_name}' collection...")
        qdrant_client.delete_collection(collection_name=collection_name)
        time.sleep(3)
        collection_exists = False

    if not collection_exists:
        # FIXED: Confirmed 768 dimensions fits gemini-embedding-001 output_dimensionality standard config
        print(f"⚙️ Provisioning a fresh vector bucket named '{collection_name}' (Configured for 768 dims)...")
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=768, distance=Distance.COSINE),
        )
        start_index = 0
        print("🚀 Starting a clean, undivided ingestion run for all records...")
    else:
        collection_info = qdrant_client.get_collection(collection_name=collection_name)
        start_index = collection_info.points_count

        if start_index >= total_records:
            print(f"🎉 Your Qdrant Cloud collection is already complete with all {start_index} records!")
            return
        print(f"⏭️ Smart Checkpoint Detected: Found {start_index} vectors in the cloud. Resuming automatically from row {start_index}...")

    print("🚀 Step D: Converting sentences to math vectors via optimized 25-row batches...")

    batch_size = 25

    for i in range(start_index, total_records, batch_size):
        batch = text_list[i : i + batch_size]
        batch_texts = [text for _, text in batch]

        try:
            # FIXED: Using standard model with target dimension project properties defined explicitly
            response = ai_client.models.embed_content(
                model="gemini-embedding-001",
                contents=batch_texts,
                config=types.EmbedContentConfig(output_dimensionality=768)
            )

            points = []
            for sub_idx, (index, text) in enumerate(batch):
                vector = response.embeddings[sub_idx].values
                points.append(
                    PointStruct(
                        id=index,
                        vector=vector,
                        payload={"page_content": text, "source": file_path}
                    )
                )

            qdrant_client.upsert(collection_name=collection_name, points=points)
            print(f"📦 Upload progress: Completed {min(i + batch_size, total_records)} / {total_records} rows...")
            time.sleep(4.0)

        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print(f"\n⚠️ Rate limit touched at batch index {i}. Pausing for 75 seconds...")
                time.sleep(75)

                response = ai_client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=batch_texts,
                    config=types.EmbedContentConfig(output_dimensionality=768)
                )
                points = []
                for sub_idx, (index, text) in enumerate(batch):
                    vector = response.embeddings[sub_idx].values
                    points.append(PointStruct(id=index, vector=vector, payload={"page_content": text, "source": file_path}))

                qdrant_client.upsert(collection_name=collection_name, points=points)
                print(f"📦 Upload progress: Completed {min(i + batch_size, total_records)} / {total_records} rows (Recovered)...")
                time.sleep(4.0)
            else:
                raise e

    print(f"\n🎉 SUCCESS! All {total_records} IMDB records are securely stored inside your aligned Qdrant Cloud Cluster!")

# --- EXECUTE THE PIPELINE WITH FORCE RESET OVERRIDE ---
file_name = "imdb_top_1000.csv"
run_phase1_unified_pipeline(file_path=file_name, collection_name="imdb_movie_dataset", force_reset=False)


📖 Step A: Verifying data file presence...
🎬 Total rows detected in dataset: 1000 movies.
🤖 Step B: Translating spreadsheet rows into descriptive text sentences...
🌐 Step C: Establishing connection to your Qdrant Cloud Cluster...
🎉 Your Qdrant Cloud collection is already complete with all 1000 records!


In [7]:
%%writefile app.py
import os
import re
import time
import uuid
import operator
from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import streamlit as st

# Core Framework Components
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Native Google GenAI, Qdrant & Langfuse Stack
from google import genai
from google.genai import types
from qdrant_client import QdrantClient
from langfuse.langchain import CallbackHandler

# =====================================================================
# PHASE 1 & 2 CONFIGURATION: ESTABLISH CREDENTIALS VIA USERDATA / ENV
# =====================================================================
st.set_page_config(page_title="Purwadhika Movie AI V2", page_icon="🎬", layout="wide")

def get_secret(key_name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(key_name, default)

GOOGLE_API_KEY = get_secret("GEMINI_API_KEY")
QDRANT_URL = get_secret("QDRANT_URL")
QDRANT_API_KEY = get_secret("QDRANT_API_KEY")
LANGFUSE_PUBLIC_KEY = get_secret("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = get_secret("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = get_secret("LANGFUSE_HOST", "https://langfuse.com")

# Synchronize Global Tracking Variables for LangChain / Langfuse
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

# Core LLM Engine with an absolute factuality lock (temperature 0.0)
@st.cache_resource
def init_clients():
    llm_engine = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        temperature=0.0,
        google_api_key=GOOGLE_API_KEY,
        thinking_budget=0,
        max_retries=2,
        timeout=None
    )
    native_ai = genai.Client(api_key=GOOGLE_API_KEY, http_options={"api_version": "v1"})
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    return llm_engine, native_ai, qdrant

llm, ai_client, qdrant_client = init_clients()


# Turn-level telemetry tracker for the System Inspector
TURN_TELEMETRY = {
    "rag_source": None,
    "input_tokens": 0,
    "output_tokens": 0
}

def track_tokens(ai_msg):
    """Extracts token usage metadata and writes directly to Streamlit session_state."""
    meta = getattr(ai_msg, "usage_metadata", None) or {}
    in_tok = meta.get("input_tokens", 0)
    out_tok = meta.get("output_tokens", 0)
    if in_tok == 0 and out_tok == 0:
        out_tok = max(1, len(str(ai_msg.content)) // 4)
        in_tok = out_tok * 3

    TURN_TELEMETRY["input_tokens"] += in_tok
    TURN_TELEMETRY["output_tokens"] += out_tok

    if "usage_metrics" in st.session_state:
        st.session_state.usage_metrics["input_tokens"] += in_tok
        st.session_state.usage_metrics["output_tokens"] += out_tok
        st.session_state.usage_metrics["total_tokens"] = (
            st.session_state.usage_metrics["input_tokens"] + st.session_state.usage_metrics["output_tokens"]
        )


# Step 3: Define the Shared Conversation State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_agent: str
    rag_source: dict | None
    input_tokens: int
    output_tokens: int


def parse_gemini_response(response_message) -> str:
    content = response_message.content
    if isinstance(content, list):
        text_parts = [
            part["text"] if isinstance(part, dict) and "text" in part else str(part)
            for part in content
        ]
        return "".join(text_parts)
    return str(content)

# =====================================================================
# STEP 4: NATIVE VECTOR DATA RETRIEVAL TOOL API
# =====================================================================
@tool
def retrieve_movie_context(search_query: str) -> str:
    """Queries the 768-dimensional Qdrant database using native Gemini embeddings."""
    try:
        if not search_query.strip():
            return "NO_CONTEXT_FOUND"

        # Generate 768-d embedding vector
        response = ai_client.models.embed_content(
            model="gemini-embedding-001",
            contents=search_query,
            config=types.EmbedContentConfig(output_dimensionality=768)
        )

        if isinstance(response.embeddings, list) and len(response.embeddings) > 0:
            query_vector = response.embeddings[0].values
        elif hasattr(response.embeddings, "values"):
            query_vector = response.embeddings.values
        else:
            query_vector = response.embeddings

        collection_name = "imdb_movie_dataset"
        query_response = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_vector,
            with_payload=True,
            limit=3
        )

        contexts = []
        top_hit_meta = None

        for idx, hit in enumerate(query_response.points):
            if hit.payload:
                page_text = hit.payload.get("page_content", "")
                contexts.append(page_text)
                if idx == 0:
                    title_match = re.search(r"(?:Title|Movie|Series_Title)\s*:\s*([^\n|]+)", page_text, re.IGNORECASE)
                    doc_title = title_match.group(1).strip() if title_match else f"IMDb Record #{hit.id}"
                    genre_match = re.search(r"(?:Genre|Category)\s*:\s*([^\n|]+)", page_text, re.IGNORECASE)
                    doc_category = genre_match.group(1).strip() if genre_match else "IMDb Dataset"

                    top_hit_meta = {
                        "document_name": doc_title,
                        "category": doc_category,
                        "database_id": str(hit.id),
                        "collection": collection_name,
                        "similarity_score": f"{hit.score:.4f}",
                        "snippet": page_text
                    }

        # Fallback string scanning sequence
        clean_keywords = re.sub(
            r"\b(what|is|the|movie|about|show|me|find|certificate|rating|run|time|director)\b",
            "",
            search_query,
            flags=re.IGNORECASE
        ).strip()

                # Only scan via scroll if vector search returned nothing or low similarity (< 0.75)
        top_score = query_response.points[0].score if query_response.points else 0.0
        if (not contexts or top_score < 0.75) and len(clean_keywords) > 2:
            scroll_results, _ = qdrant_client.scroll(
                collection_name=collection_name,
                limit=50,
                with_payload=True
            )
            for record in scroll_results:
                payload_text = record.payload.get("page_content", "")
                if clean_keywords.lower() in payload_text.lower() and payload_text not in contexts:
                    contexts.insert(0, payload_text)
                    title_match = re.search(r"(?:Title|Movie|Series_Title)\s*:\s*([^\n|]+)", payload_text, re.IGNORECASE)
                    doc_title = title_match.group(1).strip() if title_match else f"IMDb Record #{record.id}"
                    top_hit_meta = {
                        "document_name": doc_title,
                        "category": "IMDb Keyword Match",
                        "database_id": str(record.id),
                        "collection": collection_name,
                        "similarity_score": "1.0000 (Exact Keyword)",
                        "snippet": payload_text
                    }
                    break

        if not contexts:
            return "NO_CONTEXT_FOUND"

        TURN_TELEMETRY["rag_source"] = top_hit_meta
        st.session_state.last_rag_source = top_hit_meta
        return "\n\n--- Movie Record Match ---\n".join(contexts[:3])

    except Exception as e:
        return f"CRITICAL DATABASE ERROR: {str(e)}"

# =====================================================================
# AGENT NODES DESIGN MATRIX (STEPS 5, 6, 7)
# =====================================================================
def extract_token_counts(ai_msg) -> tuple[int, int]:
    meta = getattr(ai_msg, "usage_metadata", None) or {}
    in_tok = meta.get("input_tokens", 0)
    out_tok = meta.get("output_tokens", 0)
    if in_tok == 0 and out_tok == 0:
        out_tok = max(1, len(str(ai_msg.content)) // 4)
        in_tok = out_tok * 3
    return in_tok, out_tok


def movie_expert_node(state: AgentState):
    messages = state["messages"]
    user_queries = [str(m.content) for m in messages if isinstance(m, HumanMessage)]
    last_query = user_queries[-1] if user_queries else ""

    clean_lower = re.sub(r"[^\w\s]", "", last_query.lower())
    words = clean_lower.split()
    single_word_triggers = {"it", "its", "that", "this", "same"}
    phrase_triggers = ("that movie", "this movie", "the movie", "same movie")

    needs_context = (
        any(w in single_word_triggers for w in words)
        or any(p in clean_lower for p in phrase_triggers)
        or len(words) <= 5
    )

    search_query = f"{user_queries[-2]} - {last_query}" if (len(user_queries) > 1 and needs_context) else last_query

    # Reset and run retrieval directly
    TURN_TELEMETRY["rag_source"] = None
    tool_output = retrieve_movie_context.invoke({"search_query": search_query})
    retrieved_meta = TURN_TELEMETRY["rag_source"]

    # If retrieval returned an error or nothing, surface it clearly in the inspector
    if not retrieved_meta and tool_output != "NO_CONTEXT_FOUND":
        retrieved_meta = {
            "document_name": "Retrieval Debug Info",
            "category": "Tool Output",
            "database_id": "N/A",
            "collection": "imdb_movie_dataset",
            "similarity_score": "0.0000",
            "snippet": str(tool_output)
        }

    system_prompt = SystemMessage(
        content=f"You are a strict movie recommendation researcher. Extract the requested values "
                f"using ONLY the explicit movie dataset facts provided below.\n"
                f"ANTI-HALLUCINATION GUARDRAILS:\n"
                f"1. If the context is 'NO_CONTEXT_FOUND' or does not mention the requested movie, explicitly state: "
                f"'I am sorry, but that information is not available in my database records.'\n"
                f"2. Do NOT extrapolate or guess outside the retrieved data records.\n\n"
                f"RETRIEVED MOVIE CONTEXT:\n{tool_output}"
    )
    final_response = llm.invoke([system_prompt] + list(messages))
    in_tok, out_tok = extract_token_counts(final_response)

    return {
        "messages": [AIMessage(content=parse_gemini_response(final_response))],
        "next_agent": "supervisor",
        "rag_source": retrieved_meta,
        "input_tokens": in_tok,
        "output_tokens": out_tok
    }


def chit_chat_node(state: AgentState):
    system_prompt = SystemMessage(
        content="You are a friendly assistant and receptionist. Handle basic greetings, departures, and pleasantries.\n"
                "If the user asks about movie specifications, redirect them to the Movie Expert node."
    )
    response = llm.invoke([system_prompt] + list(state["messages"]))
    in_tok, out_tok = extract_token_counts(response)
    return {
        "messages": [AIMessage(content=parse_gemini_response(response))],
        "next_agent": "supervisor",
        "rag_source": None,
        "input_tokens": in_tok,
        "output_tokens": out_tok
    }



class RouterDecision(BaseModel):
    next_agent: Literal["movie_expert", "chit_chat", "FINISH"] = Field(description="Next destination node flag.")


def supervisor_node(state: AgentState):
    messages = state["messages"]
    if messages and isinstance(messages[-1], AIMessage):
        return {"next_agent": "FINISH"}

    user_queries = [str(m.content) for m in messages if isinstance(m, HumanMessage)]
    last_user_query = user_queries[-1].strip() if user_queries else ""

    # Only route pure greetings/pleasantries to chit_chat; route everything else to movie_expert
    greetings = {"hi", "hello", "hey", "halo", "thanks", "thank you", "bye", "good morning", "good evening"}
    clean_q = re.sub(r"[^\w\s]", "", last_user_query.lower()).strip()

    if clean_q in greetings:
        return {"next_agent": "chit_chat"}

    return {"next_agent": "movie_expert"}




#=====================================================================


#STEP 8: MAP GRAPH PATH NETWORK EDGES


#=====================================================================

@st.cache_resource
def build_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("movie_expert", movie_expert_node)
    workflow.add_node("chit_chat", chit_chat_node)
    workflow.add_edge(START, "supervisor")

    def route_next(state: AgentState):
        return END if state["next_agent"] == "FINISH" else state["next_agent"]

    workflow.add_conditional_edges(
        "supervisor",
        route_next,
        {"movie_expert": "movie_expert", "chit_chat": "chit_chat", END: END}
    )
    workflow.add_edge("movie_expert", "supervisor")
    workflow.add_edge("chit_chat", "supervisor")
    return workflow.compile(checkpointer=MemorySaver())


app = build_graph()


#=====================================================================


#PHASE 3: STREAMLIT CONVERSATIONAL INTERFACE & SYSTEM INSPECTOR


#=====================================================================

st.title("🎬 Purwadhika V2: Verified Core Multi-Agent Engine")
st.caption("Context-Preserving Memory Framework connected to Qdrant Cloud Storage & Langfuse Observability.")

#Initialize Session States

if "chat_history_state" not in st.session_state:
  st.session_state.chat_history_state = []
if "session_uuid" not in st.session_state:
  st.session_state.session_uuid = str(uuid.uuid4())
if "last_rag_source" not in st.session_state:
  st.session_state.last_rag_source = None
if "usage_metrics" not in st.session_state:
  st.session_state.usage_metrics = {
"input_tokens": 0,
"output_tokens": 0,
"total_tokens": 0,
"estimated_cost_usd": 0.0,
"execution_latency_sec": 0.0
}
col_chat, col_details = st.columns([2, 1.5])

#Left Side: Active Live Turn Container

with col_chat:
    st.subheader("💬 Active Conversation")
    chat_container = st.container(height=500)

    # Render history trail safely by extracting text strings
    with chat_container:
        for msg in st.session_state.chat_history_state:
            role_type = "user" if isinstance(msg, HumanMessage) else "assistant"
            with st.chat_message(role_type):
                st.write(msg.content)

    if user_input := st.chat_input("Inquire regarding movie databases here...", key="movie_rag_chat_input"):
        with chat_container:
            with st.chat_message("user"):
                st.write(user_input)

        # Reset turn telemetry & session metrics before running the graph
        TURN_TELEMETRY["rag_source"] = None
        TURN_TELEMETRY["input_tokens"] = 0
        TURN_TELEMETRY["output_tokens"] = 0
        st.session_state.last_rag_source = None
        st.session_state.usage_metrics = {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": 0.0,
            "execution_latency_sec": 0.0
        }

        # Initialize background tracing metrics using Langfuse if public key exists
        callbacks = []
        if os.environ.get("LANGFUSE_PUBLIC_KEY"):
            callbacks.append(CallbackHandler(public_key=os.environ.get("LANGFUSE_PUBLIC_KEY")))

        new_message = HumanMessage(content=user_input)
        st.session_state.chat_history_state.append(new_message)

        # Execute LangGraph engine passing complete message context history arrays
        with chat_container:
            with st.chat_message("assistant"):
                with st.spinner("Processing framework nodes..."):
                    start_time = time.perf_counter()
                    try:
                        output_bundle = app.invoke(
                            {
                                "messages": list(st.session_state.chat_history_state),
                                "rag_source": None,
                                "input_tokens": 0,
                                "output_tokens": 0
                            },
                            config={
                                "configurable": {"thread_id": st.session_state.session_uuid},
                                "callbacks": callbacks
                            }
                        )
                        elapsed_sec = round(time.perf_counter() - start_time, 2)

                        final_response = output_bundle["messages"][-1]
                        st.write(final_response.content)
                        st.session_state.chat_history_state.append(final_response)

                        # Read telemetry directly from LangGraph output_bundle
                        st.session_state.last_rag_source = output_bundle.get("rag_source")
                        in_t = output_bundle.get("input_tokens", 0)
                        out_t = output_bundle.get("output_tokens", 0)
                        tot_t = in_t + out_t
                        est_cost = (in_t * 0.000000075) + (out_t * 0.0000003)

                        st.session_state.usage_metrics = {
                            "input_tokens": in_t,
                            "output_tokens": out_t,
                            "total_tokens": tot_t,
                            "estimated_cost_usd": est_cost,
                            "execution_latency_sec": elapsed_sec
                        }
                        st.rerun()

                    except Exception as api_err:
                        error_msg = str(api_err)
                        if "504" in error_msg or "DEADLINE_EXCEEDED" in error_msg:
                            st.error("⚠️ Request timed out (504). The Gemini server took too long to respond—please click send again.")
                        elif "503" in error_msg or "UNAVAILABLE" in error_msg:
                            st.error("⚠️ Gemini API is experiencing high traffic spikes. Please wait a moment and try submitting your request again.")
                        elif "429" in error_msg:
                            st.error("⚠️ Daily rate limit reached. Please verify your API key quota.")
                        else:
                            st.error(f"❌ Execution Error: {error_msg}")




#Right Side: Houses all 3 Drop-Down Accordions exclusively

with col_details:
    header_col, btn_col = st.columns([3, 1])
    with header_col:
        st.subheader("🛠️ System Inspector")
    with btn_col:
        if st.button("🗑️ Reset", use_container_width=True):
            st.session_state.session_uuid = str(uuid.uuid4())
            st.session_state.chat_history_state = []
            st.session_state.last_rag_source = None
            st.session_state.usage_metrics = {
                "input_tokens": 0,
                "output_tokens": 0,
                "total_tokens": 0,
                "estimated_cost_usd": 0.0,
                "execution_latency_sec": 0.0
            }
            st.rerun()

    # DROP-DOWN 1: Complete Historical Log
    with st.expander("📜 Click to view Complete Chat History Log", expanded=False):
        st.caption(f"Session UUID: `{st.session_state.session_uuid}`")
        if not st.session_state.chat_history_state:
            st.write("No messages recorded in this session yet.")
        else:
            for idx, msg in enumerate(st.session_state.chat_history_state):
                role_label = "🧑 User" if isinstance(msg, HumanMessage) else "🤖 Assistant"
                st.markdown(f"**[{idx + 1}] {role_label}:** {msg.content}")
                st.divider()

    # DROP-DOWN 2: Vector Database Evidence
    with st.expander("🔍 Click to view Vector Database Evidence", expanded=True):
        if st.session_state.last_rag_source:
            src = st.session_state.last_rag_source
            st.markdown(f"**📄 Document Name:** `{src['document_name']}`")
            st.markdown(f"**🗂️ Category Type:** `{src['category']}`")
            st.markdown(f"**🆔 Vector Payload ID:** `{src['database_id']}`")
            st.markdown(f"**🌐 Qdrant Collection:** `{src['collection']}`")
            st.markdown(f"**🎯 Math Similarity Match:** `{src['similarity_score']}`")
            st.markdown("**📝 Ingested Raw Text Chunk:**")
            st.code(src["snippet"], language="text")
        else:
            st.write("No vector retrieval occurred for this turn.")

    # DROP-DOWN 3: Token Usage Details
    with st.expander("⚙️ Click to view Token Usage & Metrics", expanded=True):
        st.metric(label="📥 Input Tokens", value=st.session_state.usage_metrics["input_tokens"])
        st.metric(label="📤 Output Tokens", value=st.session_state.usage_metrics["output_tokens"])
        st.metric(label="📊 Total Tokens Used", value=st.session_state.usage_metrics["total_tokens"])
        st.divider()
        st.markdown(f"**Turn Cost:** `${st.session_state.usage_metrics['estimated_cost_usd']:.5f}`")
        st.markdown(f"**Graph Latency:** `{st.session_state.usage_metrics['execution_latency_sec']}s`")


Overwriting app.py


In [8]:
import os
import subprocess
import time
import signal
from google.colab import userdata

print("🔄 Cleaning previous server environments...")
try:
    pids = subprocess.check_output(["pgrep", "-f", "streamlit run app.py"]).decode().split()
    for pid in pids:
        os.kill(int(pid), signal.SIGTERM)
    time.sleep(2)
except Exception:
    pass

# 1. Extract keys dynamically from Colab Secrets
gemini_key = userdata.get('GEMINI_API_KEY')
qdrant_url = userdata.get('QDRANT_URL')
qdrant_api_key = userdata.get('QDRANT_API_KEY')

try:
    langfuse_pub = userdata.get('LANGFUSE_PUBLIC_KEY') or ''
except Exception:
    langfuse_pub = ''

try:
    langfuse_sec = userdata.get('LANGFUSE_SECRET_KEY') or ''
except Exception:
    langfuse_sec = ''

try:
    langfuse_host = userdata.get('LANGFUSE_HOST') or 'https://cloud.langfuse.com'
except Exception:
    langfuse_host = 'https://cloud.langfuse.com'

# 2. Establish environment variables for the Streamlit subprocess
env = {
    **os.environ,
    "GEMINI_API_KEY": gemini_key,
    "GOOGLE_API_KEY": gemini_key,
    "QDRANT_URL": qdrant_url,
    "QDRANT_API_KEY": qdrant_api_key,
    "LANGFUSE_PUBLIC_KEY": langfuse_pub,
    "LANGFUSE_SECRET_KEY": langfuse_sec,
    "LANGFUSE_HOST": langfuse_host
}

# 3. Spin up Streamlit and save logs to streamlit.log for easy debugging
print("🚀 Deploying background instances on Port 8520...")
log_file = open("streamlit.log", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8520", "--server.headless", "true"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)
time.sleep(5)

print("\n🎉 Server Online!")
print("🔗 CLICK THE SERVEO TUNNEL LINK BELOW TO CHAT NATIVELY:\n")

# 4. Connect to Serveo remote forward tunnel bridge mapping port 8520
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8520 serveo.net


🔄 Cleaning previous server environments...
🚀 Deploying background instances on Port 8520...

🎉 Server Online!
🔗 CLICK THE SERVEO TUNNEL LINK BELOW TO CHAT NATIVELY:

Forwarding HTTP traffic from https://a7007aab284746c9-136-85-185-29.serveousercontent.com
Tip (1): Add your SSH key to reserve names and keep URLs stable. Pro removes the browser warning page: https://console.serveo.net/settings?n=1&src=ssh_nudge&v=A
